In [1]:
import pandas as pd
import random
from datetime import datetime, timedelta

random.seed(42)

NUM_CLEAN_ORDERS = 60
NUM_MISSING_PAYMENT = 6
NUM_AMOUNT_MISMATCH = 6
NUM_DUPLICATE_PAYMENT = 4
NUM_LATE_PAYMENT = 4

names = ["Aarav", "Vivaan", "Aditi", "Diya", "Kabir", "Meera", "Rohan",
         "Ishaan", "Ananya", "Sara", "Arjun", "Priya", "Karan", "Neha"]

orders = []
payments = []

order_counter = 1000
base_date = datetime(2026, 8, 1)

def new_order_id():
    global order_counter
    order_counter += 1
    return f"ORD{order_counter}"

for _ in range(NUM_CLEAN_ORDERS):
    oid = new_order_id()
    amount = round(random.uniform(200, 5000), 2)
    order_date = base_date + timedelta(days=random.randint(0, 20))
    pay_date = order_date + timedelta(days=random.randint(0, 2))
    orders.append({"order_id": oid, "customer_name": random.choice(names),
        "amount": amount, "order_date": order_date.strftime("%Y-%m-%d"), "status": "placed"})
    payments.append({"payment_id": oid, "amount_received": amount,
        "payment_date": pay_date.strftime("%Y-%m-%d"), "status": "success"})

for _ in range(NUM_MISSING_PAYMENT):
    oid = new_order_id()
    amount = round(random.uniform(200, 5000), 2)
    order_date = base_date + timedelta(days=random.randint(0, 20))
    orders.append({"order_id": oid, "customer_name": random.choice(names),
        "amount": amount, "order_date": order_date.strftime("%Y-%m-%d"), "status": "placed"})

for _ in range(NUM_AMOUNT_MISMATCH):
    oid = new_order_id()
    amount = round(random.uniform(500, 5000), 2)
    order_date = base_date + timedelta(days=random.randint(0, 20))
    pay_date = order_date + timedelta(days=random.randint(0, 3))
    shortfall = round(random.uniform(20, 200), 2)
    orders.append({"order_id": oid, "customer_name": random.choice(names),
        "amount": amount, "order_date": order_date.strftime("%Y-%m-%d"), "status": "placed"})
    payments.append({"payment_id": oid, "amount_received": round(amount - shortfall, 2),
        "payment_date": pay_date.strftime("%Y-%m-%d"), "status": "success"})

for _ in range(NUM_DUPLICATE_PAYMENT):
    oid = new_order_id()
    amount = round(random.uniform(200, 3000), 2)
    order_date = base_date + timedelta(days=random.randint(0, 20))
    pay_date = order_date + timedelta(days=random.randint(0, 2))
    orders.append({"order_id": oid, "customer_name": random.choice(names),
        "amount": amount, "order_date": order_date.strftime("%Y-%m-%d"), "status": "placed"})
    payments.append({"payment_id": oid, "amount_received": amount,
        "payment_date": pay_date.strftime("%Y-%m-%d"), "status": "success"})
    payments.append({"payment_id": oid, "amount_received": amount,
        "payment_date": pay_date.strftime("%Y-%m-%d"), "status": "success"})

for _ in range(NUM_LATE_PAYMENT):
    oid = new_order_id()
    amount = round(random.uniform(200, 3000), 2)
    order_date = base_date + timedelta(days=random.randint(0, 15))
    pay_date = order_date + timedelta(days=random.randint(14, 25))
    orders.append({"order_id": oid, "customer_name": random.choice(names),
        "amount": amount, "order_date": order_date.strftime("%Y-%m-%d"), "status": "placed"})
    payments.append({"payment_id": oid, "amount_received": amount,
        "payment_date": pay_date.strftime("%Y-%m-%d"), "status": "success"})

orders_df = pd.DataFrame(orders)
payments_df = pd.DataFrame(payments)
orders_df = orders_df.sample(frac=1, random_state=1).reset_index(drop=True)
payments_df = payments_df.sample(frac=1, random_state=1).reset_index(drop=True)

orders_df.to_csv("orders.csv", index=False)
payments_df.to_csv("payments.csv", index=False)

print(f"Generated {len(orders_df)} orders and {len(payments_df)} payments.")
print("Saved to orders.csv and payments.csv")
print("\nSample orders:")
print(orders_df.head())
print("\nSample payments:")
print(payments_df.head())

Generated 80 orders and 78 payments.
Saved to orders.csv and payments.csv

Sample orders:
  order_id customer_name   amount  order_date  status
0  ORD1064          Diya  4336.03  2026-08-08  placed
1  ORD1028         Priya  3775.95  2026-08-18  placed
2  ORD1032        Vivaan  3211.74  2026-08-14  placed
3  ORD1070         Rohan  4390.93  2026-08-02  placed
4  ORD1047         Aditi   803.65  2026-08-16  placed

Sample payments:
  payment_id  amount_received payment_date   status
0    ORD1074          1718.15   2026-08-11  success
1    ORD1080          1373.83   2026-08-29  success
2    ORD1045          4414.45   2026-08-03  success
3    ORD1028          3775.95   2026-08-19  success
4    ORD1070          4302.71   2026-08-03  success


In [2]:
"""
Step 2: Match orders against payments, and figure out which ones
don't line up (and why). This is the core reconciliation logic.
"""

import pandas as pd

# Load the datasets you created on Day 1
orders_df = pd.read_csv("orders.csv")
payments_df = pd.read_csv("payments.csv")

# --- Step A: Find duplicate payments first (same payment_id appearing twice) ---
duplicate_ids = payments_df[payments_df.duplicated(subset="payment_id", keep=False)]["payment_id"].unique()

# Drop duplicates for the main merge, but we'll report them separately
payments_deduped = payments_df.drop_duplicates(subset="payment_id", keep="first")

# --- Step B: Merge orders and payments on their shared ID ---
merged = orders_df.merge(
    payments_deduped,
    left_on="order_id",
    right_on="payment_id",
    how="left",              # keep every order, even if no payment matches
    indicator=True             # adds a column telling us if it matched or not
)

# --- Step C: Classify every row into a category ---
def classify(row):
    if row["order_id"] in duplicate_ids:
        return "duplicate_payment"
    if row["_merge"] == "left_only":
        return "missing_payment"
    if pd.notna(row["amount_received"]) and abs(row["amount"] - row["amount_received"]) > 0.01:
        return "amount_mismatch"
    if pd.notna(row["payment_date"]):
        order_date = pd.to_datetime(row["order_date"])
        pay_date = pd.to_datetime(row["payment_date"])
        if (pay_date - order_date).days > 10:
            return "late_payment"
    return "matched"

merged["exception_type"] = merged.apply(classify, axis=1)

# --- Step D: Calculate impact ($ at risk) for ranking later ---
def calc_impact(row):
    if row["exception_type"] == "matched":
        return 0
    if row["exception_type"] == "amount_mismatch":
        return abs(row["amount"] - row["amount_received"])
    return row["amount"]  # full order amount is "at risk" for missing/duplicate/late

merged["impact_amount"] = merged.apply(calc_impact, axis=1)

# --- Step E: Split into matched vs exceptions, and calculate match rate ---
matched = merged[merged["exception_type"] == "matched"]
exceptions = merged[merged["exception_type"] != "matched"].copy()
exceptions = exceptions.sort_values("impact_amount", ascending=False)

match_rate = round(len(matched) / len(merged) * 100, 1)

# --- Step F: Print the results ---
print(f"===== RECONCILIATION SUMMARY =====")
print(f"Total orders processed: {len(merged)}")
print(f"Matched cleanly: {len(matched)}")
print(f"Exceptions found: {len(exceptions)}")
print(f"Match rate: {match_rate}%")
print()
print("Exception breakdown by type:")
print(exceptions["exception_type"].value_counts())
print()
print("===== TOP 10 EXCEPTIONS BY $ IMPACT =====")
print(exceptions[["order_id", "customer_name", "amount", "amount_received",
                   "exception_type", "impact_amount"]].head(10).to_string(index=False))

# Save exceptions to a CSV so we can use it in Day 3 (AI explanations)
exceptions.to_csv("exceptions.csv", index=False)
print("\nSaved full exception list to exceptions.csv")

===== RECONCILIATION SUMMARY =====
Total orders processed: 80
Matched cleanly: 60
Exceptions found: 20
Match rate: 75.0%

Exception breakdown by type:
exception_type
missing_payment      6
amount_mismatch      6
duplicate_payment    4
late_payment         4
Name: count, dtype: int64

===== TOP 10 EXCEPTIONS BY $ IMPACT =====
order_id customer_name  amount  amount_received    exception_type  impact_amount
 ORD1061         Arjun 4346.42              NaN   missing_payment        4346.42
 ORD1064          Diya 4336.03              NaN   missing_payment        4336.03
 ORD1062        Vivaan 3336.59              NaN   missing_payment        3336.59
 ORD1066        Ishaan 2225.03              NaN   missing_payment        2225.03
 ORD1075         Aditi 1835.70          1835.70 duplicate_payment        1835.70
 ORD1079        Vivaan 1820.97          1820.97      late_payment        1820.97
 ORD1074         Aarav 1718.15          1718.15 duplicate_payment        1718.15
 ORD1080          Diya 13

In [3]:
!pip install google-generativeai -q

In [4]:
"""
Step 3: Use Gemini (free) to explain each top exception in plain English
and suggest a next action. This is what makes the project an AI agent.
"""

import pandas as pd
import google.generativeai as genai
from getpass import getpass

# You'll be asked to paste your key when this cell runs
api_key = getpass("Paste your Google API key here: ")
genai.configure(api_key=api_key)
model = genai.GenerativeModel("gemini-3.6-flash")

# Load the exceptions you found in Day 2
exceptions = pd.read_csv("exceptions.csv")

# Only process the top 5 (by impact) to keep this fast
top_exceptions = exceptions.head(5)

results = []

for _, row in top_exceptions.iterrows():
    prompt = f"""You are a finance operations assistant. Here is a reconciliation exception:

Order ID: {row['order_id']}
Customer: {row['customer_name']}
Order amount: ₹{row['amount']}
Amount received: {row['amount_received'] if pd.notna(row['amount_received']) else 'None (no payment found)'}
Exception type: {row['exception_type']}
Impact amount: ₹{row['impact_amount']}

Respond in exactly this format, nothing else:
EXPLANATION: <one sentence explaining what likely happened, in plain English>
ACTION: <one short suggested next step, like a brief email or an internal action>"""

    response = model.generate_content(prompt)
    ai_text = response.text

    # Split the response into explanation and action
    explanation = ai_text.split("EXPLANATION:")[1].split("ACTION:")[0].strip()
    action = ai_text.split("ACTION:")[1].strip()

    results.append({
        "order_id": row["order_id"],
        "exception_type": row["exception_type"],
        "impact_amount": row["impact_amount"],
        "ai_explanation": explanation,
        "ai_suggested_action": action
    })

    print(f"--- {row['order_id']} ({row['exception_type']}, ₹{row['impact_amount']} at risk) ---")
    print(f"Explanation: {explanation}")
    print(f"Suggested action: {action}")
    print()

# Save this for Day 4 (final report)
results_df = pd.DataFrame(results)
results_df.to_csv("ai_explained_exceptions.csv", index=False)
print("Saved to ai_explained_exceptions.csv")

/usr/local/lib/python3.13/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


Paste your Google API key here: ··········
--- ORD1061 (missing_payment, ₹4346.42 at risk) ---
Explanation: Order ORD1061 was recorded for Arjun, but no corresponding payment of ₹4346.42 was captured or received in the system.
Suggested action: Contact Arjun to request proof of payment or re-initiate the payment process, and check the payment gateway logs for failed transactions.

--- ORD1064 (missing_payment, ₹4336.03 at risk) ---
Explanation: Order ORD1064 was processed for Diya, but the payment of ₹4336.03 was not received or failed to record in the system.
Suggested action: Check the payment gateway logs for failed transactions or email the customer to request payment verification.

--- ORD1062 (missing_payment, ₹3336.59 at risk) ---
Explanation: Order ORD1062 was placed by Vivaan for ₹3336.59, but no corresponding payment transaction was recorded in the system.
Suggested action: Contact Vivaan to request payment confirmation or verify the transaction status with the payment gatewa

In [5]:
"""
Step 4 (v2): Combine everything into one polished, ledger-style HTML report.
"""

import pandas as pd

orders_df = pd.read_csv("orders.csv")
payments_df = pd.read_csv("payments.csv")
exceptions = pd.read_csv("exceptions.csv")
ai_explained = pd.read_csv("ai_explained_exceptions.csv")

total_orders = len(orders_df)
total_exceptions = len(exceptions)
matched_count = total_orders - total_exceptions
match_rate = round(matched_count / total_orders * 100, 1)
total_amount_at_risk = round(exceptions["impact_amount"].sum(), 2)

TYPE_LABELS = {
    "missing_payment": "Missing Payment",
    "amount_mismatch": "Amount Mismatch",
    "duplicate_payment": "Duplicate Payment",
    "late_payment": "Late Payment",
}

html = f"""
<!DOCTYPE html>
<html>
<head>
<meta charset="UTF-8">
<title>AI Finance Controller — Reconciliation Ledger</title>
<link rel="preconnect" href="https://fonts.googleapis.com">
<link href="https://fonts.googleapis.com/css2?family=IBM+Plex+Serif:wght@400;600;700&family=IBM+Plex+Sans:wght@400;500;600&family=IBM+Plex+Mono:wght@400;500;600&display=swap" rel="stylesheet">
<style>
    :root {{
        --bg: #0B1220;
        --panel: #121A2C;
        --panel-2: #17203570;
        --border: #26314A;
        --text: #E8ECF4;
        --muted: #8993AB;
        --accent: #E8A33D;
        --accent-dim: #E8A33D22;
        --risk: #E5595F;
        --risk-dim: #E5595F1a;
        --ok: #4ADE80;
    }}
    * {{ box-sizing: border-box; }}
    body {{
        font-family: 'IBM Plex Sans', sans-serif;
        background: var(--bg);
        background-image:
            linear-gradient(var(--panel-2) 1px, transparent 1px),
            linear-gradient(90deg, var(--panel-2) 1px, transparent 1px);
        background-size: 100% 32px, 32px 100%;
        color: var(--text);
        margin: 0;
        padding: 48px 24px 80px;
    }}
    .container {{ max-width: 880px; margin: 0 auto; }}

    .masthead {{ border-bottom: 1px solid var(--border); padding-bottom: 20px; margin-bottom: 32px; display: flex; justify-content: space-between; align-items: flex-end; }}
    .masthead-title {{ font-family: 'IBM Plex Serif', serif; font-weight: 700; font-size: 26px; letter-spacing: -0.01em; }}
    .masthead-title span {{ color: var(--accent); }}
    .masthead-sub {{ font-family: 'IBM Plex Mono', monospace; color: var(--muted); font-size: 12px; text-transform: uppercase; letter-spacing: 0.08em; margin-top: 4px; }}
    .masthead-stamp {{ font-family: 'IBM Plex Mono', monospace; font-size: 11px; color: var(--muted); text-align: right; line-height: 1.6; }}

    .ledger-strip {{
        display: grid;
        grid-template-columns: repeat(4, 1fr);
        border: 1px solid var(--border);
        border-radius: 4px;
        overflow: hidden;
        margin-bottom: 40px;
        background: var(--panel);
    }}
    .ledger-cell {{ padding: 20px 18px; border-right: 1px solid var(--border); }}
    .ledger-cell:last-child {{ border-right: none; }}
    .ledger-num {{ font-family: 'IBM Plex Mono', monospace; font-size: 26px; font-weight: 600; font-variant-numeric: tabular-nums; }}
    .ledger-label {{ font-size: 11px; color: var(--muted); text-transform: uppercase; letter-spacing: 0.06em; margin-top: 6px; }}
    .num-ok {{ color: var(--ok); }}
    .num-risk {{ color: var(--risk); }}
    .num-accent {{ color: var(--accent); }}

    .section-title {{
        font-family: 'IBM Plex Mono', monospace;
        font-size: 12px; text-transform: uppercase; letter-spacing: 0.1em;
        color: var(--accent); margin: 40px 0 16px;
        display: flex; align-items: center; gap: 12px;
    }}
    .section-title::after {{ content: ""; flex: 1; height: 1px; background: var(--border); }}

    .stub {{
        background: var(--panel);
        border: 1px solid var(--border);
        border-radius: 6px;
        margin-bottom: 14px;
        position: relative;
        overflow: hidden;
    }}
    .stub::before {{ content: ""; position: absolute; left: 0; top: 0; bottom: 0; width: 3px; background: var(--risk); }}
    .stub-head {{
        display: flex; justify-content: space-between; align-items: center;
        padding: 16px 20px 12px 24px; border-bottom: 1px dashed var(--border);
    }}
    .stub-id {{ font-family: 'IBM Plex Mono', monospace; font-weight: 600; font-size: 14px; }}
    .stub-type {{ font-family: 'IBM Plex Mono', monospace; font-size: 10px; text-transform: uppercase; letter-spacing: 0.06em; color: var(--risk); background: var(--risk-dim); padding: 4px 10px; border-radius: 20px; }}
    .stub-body {{ padding: 16px 24px 20px; }}
    .stub-amount {{ font-family: 'IBM Plex Mono', monospace; font-size: 20px; font-weight: 600; color: var(--risk); font-variant-numeric: tabular-nums; margin-bottom: 10px; }}
    .stub-explanation {{ color: var(--text); line-height: 1.6; font-size: 14px; margin-bottom: 14px; }}
    .stub-action {{ display: flex; gap: 10px; padding: 12px 14px; background: var(--accent-dim); border: 1px solid #E8A33D33; border-radius: 4px; font-size: 13px; line-height: 1.5; }}
    .stub-action-icon {{ color: var(--accent); font-family: 'IBM Plex Mono', monospace; font-weight: 700; flex-shrink: 0; }}

    .footer {{ margin-top: 48px; padding-top: 20px; border-top: 1px solid var(--border); font-family: 'IBM Plex Mono', monospace; font-size: 11px; color: var(--muted); text-align: center; letter-spacing: 0.04em; }}
</style>
</head>
<body>
<div class="container">

    <div class="masthead">
        <div>
            <div class="masthead-title">AI Finance <span>Controller</span></div>
            <div class="masthead-sub">Reconciliation Ledger — Orders vs Payments</div>
        </div>
        <div class="masthead-stamp">
            BATCH SIZE: {total_orders} RECORDS<br>
            RUN TYPE: SYNTHETIC TEST DATA
        </div>
    </div>

    <div class="ledger-strip">
        <div class="ledger-cell">
            <div class="ledger-num">{total_orders}</div>
            <div class="ledger-label">Orders Processed</div>
        </div>
        <div class="ledger-cell">
            <div class="ledger-num num-ok">{match_rate}%</div>
            <div class="ledger-label">Match Rate</div>
        </div>
        <div class="ledger-cell">
            <div class="ledger-num num-risk">{total_exceptions}</div>
            <div class="ledger-label">Exceptions Found</div>
        </div>
        <div class="ledger-cell">
            <div class="ledger-num num-accent">₹{total_amount_at_risk:,.0f}</div>
            <div class="ledger-label">Amount At Risk</div>
        </div>
    </div>

    <div class="section-title">Flagged Exceptions — AI Reviewed</div>
"""

for _, row in ai_explained.iterrows():
    type_label = TYPE_LABELS.get(row["exception_type"], row["exception_type"].replace("_", " ").title())
    html += f"""
    <div class="stub">
        <div class="stub-head">
            <span class="stub-id">{row['order_id']}</span>
            <span class="stub-type">{type_label}</span>
        </div>
        <div class="stub-body">
            <div class="stub-amount">₹{row['impact_amount']:,.2f}</div>
            <div class="stub-explanation">{row['ai_explanation']}</div>
            <div class="stub-action">
                <span class="stub-action-icon">→</span>
                <span>{row['ai_suggested_action']}</span>
            </div>
        </div>
    </div>
    """

html += """
    <div class="footer">GENERATED FOR RAZORPAY AI BUILDATHON · TRACK 04 · AI FINANCE CONTROLLER</div>
</div>
</body>
</html>
"""

with open("reconciliation_report.html", "w") as f:
    f.write(html)

print("Report saved to reconciliation_report.html")
print(f"\nSummary: {total_orders} orders processed, {match_rate}% match rate, "
      f"{total_exceptions} exceptions found, ₹{total_amount_at_risk:,.0f} at risk.")

Report saved to reconciliation_report.html

Summary: 80 orders processed, 75.0% match rate, 20 exceptions found, ₹26,044 at risk.
